In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
# 1. Setup and Data Preparation
# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# Normalize pixel values
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
# Reshape the data to a 1D vector (784 pixels)
x_train = x_train.reshape((len(x_train), np.prod(x_train.shape[1:])))
x_test = x_test.reshape((len(x_test), np.prod(x_test.shape[1:])))
# Filter the dataset to train on only one class (e.g., digit '3')
# This is our 'normal' data
train_normal = x_train[y_train == 3]
test_normal = x_test[y_test == 3]
# 2. Build the Autoencoder Model
# Define the size of the encoded representation
encoding_dim = 32
# Input placeholder
input_img = Input(shape=(784,))
# Encoder part
encoded = Dense(128, activation='relu')(input_img)
encoded = Dense(64, activation='relu')(encoded)
encoded = Dense(encoding_dim, activation='relu')(encoded)
# Decoder part
decoded = Dense(64, activation='relu')(encoded)
decoded = Dense(128, activation='relu')(decoded)
decoded = Dense(784, activation='sigmoid')(decoded)
# The autoencoder model
autoencoder = Model(input_img, decoded)
# Compile the autoencoder
autoencoder.compile(optimizer='adam', loss='mse')
# Print the model summary
autoencoder.summary()
# 3. Train the Autoencoder
print("Training the autoencoder...")
autoencoder.fit(train_normal, train_normal,
epochs=50,
batch_size=256,
shuffle=True,
validation_data=(test_normal, test_normal))
# 4. Detect Anomalies and Visualize Results
# Create a test set with normal and anomalous data
normal_test_data = x_test[y_test == 3]
anomalous_test_data = x_test[y_test == 7]
test_data = np.concatenate([normal_test_data[:100], anomalous_test_data[:100]])
test_labels = np.array([0] * 100 + [1] * 100) # 0 for normal, 1 for anomaly
# Get the reconstruction for the test data
reconstructions = autoencoder.predict(test_data)
# Calculate the mean squared error (reconstruction error) for each data point
mse = np.mean(np.power(test_data - reconstructions, 2), axis=1)
# Visualize the reconstruction error distribution
plt.figure(figsize=(10, 6))
plt.hist(mse[test_labels == 0], bins=50, alpha=0.7, label='Normal (Digit 3)')
plt.hist(mse[test_labels == 1], bins=50, alpha=0.7, label='Anomaly (Digit 7)')
plt.title('Reconstruction Error Distribution')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Number of Samples')
plt.legend()
plt.show()
# Set a threshold to classify anomalies
threshold = np.mean(mse[test_labels == 0]) + 2 * np.std(mse[test_labels == 0])
print(f"\nThreshold for anomaly detection: {threshold:.4f}")
print(f"Number of anomalies detected (error > threshold): {np.sum(mse > threshold)}")
print(f"Total number of test samples: {len(test_data)}")

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 784)]             0         
                                                                 
 dense (Dense)               (None, 128)               100480    
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 64)                2112      
                                                                 
 dense_4 (Dense)             (None, 128)               8320      
                                                                 
 dense_5 (Dense)             (None, 784)               101136